# Selecting Polymarket Markets for Financial Analysis

**Goal**: identify among Polymarket's ~150,000 markets those relevant to our research — macro-financial markets (Fed, inflation, GDP, geopolitics, crypto) — and build their hourly price series.

This notebook covers the first two steps of the pipeline:
1. **Selection & categorization** of relevant resolved markets
2. **Building hourly price series** from BigQuery

By the end, we'll have a clean dataset ready for crowd calibration analysis (notebook 02) and correlation tests (notebook 03).

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Adjust depending on environment (Colab or local)
# On Colab: mount GCS or upload files to data/
DATA_DIR = "../data"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Raw Data

We start from two sources:
- **`markets_snapshot.csv`** — full snapshot of ~150K Polymarket markets (526 MB), collected via the Gamma API
- **BigQuery `polymarket.trades`** — 418M historical trades extracted from the Polygon blockchain (dataset [SII-WANGZJ](https://huggingface.co/datasets/SII-WANGZJ/Polymarket_data))

The CLOB API only returns price history for **active** markets. For resolved markets (the ones we need, since we know the outcome), we have to use onchain trades.

In [ ]:
# Load the full snapshot (may take ~30s due to file size)
markets_raw = pd.read_csv(os.path.join(DATA_DIR, "markets_snapshot.csv"), low_memory=False)
print(f"Total markets: {len(markets_raw):,}")
print(f"Columns: {markets_raw.shape[1]}")

# How many are closed/resolved?
if "closed" in markets_raw.columns:
    closed = markets_raw[markets_raw["closed"].astype(str).isin(["1", "True", "true"])]
    print(f"Resolved markets: {len(closed):,} ({len(closed)/len(markets_raw)*100:.0f}%)")

## 2. Filtering Strategy

We don't keep all 150K markets — the vast majority cover sports, pop culture, weather, etc. We target **5 macro-financial categories**:

| Category | Keywords | Associated financial tickers |
|----------|----------|------------------------------|
| `fed_rates` | fed decrease, rate cut, rate hike... | TLT, ^TNX, DX-Y.NYB, SPY |
| `inflation_cpi` | monthly inflation, cpi, annual inflation... | TLT, GLD, SPY, ^TNX |
| `macro_gdp` | recession, gdp growth, gdp contraction... | SPY, QQQ, ^VIX, TLT |
| `geopolitics_trade` | tariff, trade war, sanction... | SPY, CL=F, GLD, DX-Y.NYB |
| `crypto_price` | bitcoin above/below, ethereum above/below... | BTC-USD, ETH-USD, SOL-USD |

Each category also has **exclusion keywords** to filter out noise (e.g. "fed chair" refers to the person, not interest rates).

Additional filter: **minimum volume $100K** — low-liquidity markets are not reliable signals.

In [ ]:
# Categorization rules
CATEGORY_RULES = {
    "fed_rates": {
        "include": [
            "fed decrease", "fed increase", "no change in fed",
            "interest rate", "rate cut", "rate hike",
        ],
        "exclude": [
            "fed chair", "nominate", "fire", 'say "', "say '",
            "resign", "impeach",
        ],
        "tickers": ["TLT", "^TNX", "DX-Y.NYB", "SPY"],
    },
    "inflation_cpi": {
        "include": [
            "monthly inflation", "annual inflation",
            "inflation increase", "inflation reach", "inflation get",
            "cpi ",
        ],
        "exclude": [
            "powell say", "argentina", "canada", "japan",
            "india", "uk ", "mexico", "turkey", "brazil",
        ],
        "tickers": ["TLT", "GLD", "SPY", "^TNX"],
    },
    "macro_gdp": {
        "include": [
            "recession", "gdp growth", "gdp decline",
            "gdp contraction", "negative gdp",
        ],
        "exclude": ["china gdp", "uk gdp", "euro gdp"],
        "tickers": ["SPY", "QQQ", "^VIX", "TLT"],
    },
    "geopolitics_trade": {
        "include": ["tariff", "trade war", "sanction"],
        "exclude": ["fart", "dividend", "tariff rate on china"],
        "tickers": ["SPY", "CL=F", "GLD", "DX-Y.NYB"],
    },
    "crypto_price": {
        "include": [
            "bitcoin above", "bitcoin below", "bitcoin dip",
            "bitcoin reach", "btc above", "btc below",
            "ethereum above", "ethereum below",
        ],
        "exclude": ["up or down"],
        "tickers": ["BTC-USD", "ETH-USD", "SOL-USD"],
    },
}

MIN_VOLUME_USD = 100_000


def categorize(question: str) -> str | None:
    """Categorize a market by keywords. Returns None if not relevant."""
    q = question.lower()
    for cat, rules in CATEGORY_RULES.items():
        if any(exc in q for exc in rules["exclude"]):
            continue
        if any(inc in q for inc in rules["include"]):
            return cat
    return None


def parse_outcome(outcome_prices_str) -> float | None:
    """
    Extract the real outcome from outcome_prices.
    Yes >= 0.90 -> outcome = 1, Yes <= 0.10 -> outcome = 0, otherwise ambiguous.
    """
    if pd.isna(outcome_prices_str):
        return None
    try:
        s = str(outcome_prices_str).replace("'", '"')
        prices = json.loads(s)
        yes_p = float(prices[0])
        if yes_p >= 0.90:
            return 1.0
        elif yes_p <= 0.10:
            return 0.0
        return None
    except Exception:
        return None

## 3. Running the Selection Pipeline

We apply filters in order:
1. Volume >= $100K
2. Keyword categorization (5 macro categories)
3. Clear outcome (final Yes price > 90% or < 10% — we exclude ambiguous 50/50 resolutions)

In [ ]:
# Normalize column names (Gamma API -> standardized names)
col_map = {
    "conditionId": "condition_id",
    "clobTokenIds_0": "token1",
    "clobTokenIds_1": "token2",
    "outcomes_0": "answer1",
    "outcomes_1": "answer2",
    "outcomePrices": "outcome_prices",
    "endDate": "end_date",
    "createdAt": "created_at",
}
df = markets_raw.rename(columns={k: v for k, v in col_map.items() if k in markets_raw.columns})

# Filter closed markets only
if "closed" in df.columns:
    df = df[df["closed"].astype(str).isin(["1", "True", "true"])]

# Step 1 — Minimum volume
df = df[df["volume"] >= MIN_VOLUME_USD].copy()
print(f"After volume filter >= ${MIN_VOLUME_USD:,.0f}: {len(df):,} markets")

# Step 2 — Categorization
df["category"] = df["question"].apply(categorize)
df = df.dropna(subset=["category"])
print(f"After macro/finance categorization:  {len(df):,} markets")

# Step 3 — Clear outcome
df["outcome_real"] = df["outcome_prices"].apply(parse_outcome)
df = df.dropna(subset=["outcome_real"])
df["outcome_real"] = df["outcome_real"].astype(int)
print(f"With clear outcome (>90% or <10%):   {len(df):,} markets")

# Add matched tickers
df["matched_tickers"] = df["category"].map(
    {cat: ",".join(r["tickers"]) for cat, r in CATEGORY_RULES.items()}
)

# Sort
df = df.sort_values(["category", "volume"], ascending=[True, False])

# Final columns
cols = [
    "id", "condition_id", "token1", "token2",
    "question", "event_title", "event_id",
    "category", "outcome_real", "matched_tickers",
    "volume", "outcome_prices", "created_at", "end_date",
]
cols = [c for c in cols if c in df.columns]
selected = df[cols].reset_index(drop=True)

print(f"\n{'='*60}")
print(f"TOTAL SELECTED: {len(selected)} markets")
print(f"{'='*60}")

## 4. Exploring Selected Markets

Let's look at what we kept: how many markets per category, what volumes, and concrete examples of questions.

In [ ]:
# Breakdown by category
cat_stats = (
    selected.groupby("category")
    .agg(
        n_markets=("question", "count"),
        volume_total=("volume", "sum"),
        volume_median=("volume", "median"),
        pct_yes=("outcome_real", "mean"),
    )
    .sort_values("volume_total", ascending=False)
)

cat_stats["volume_total_M"] = cat_stats["volume_total"] / 1e6
cat_stats["volume_median_K"] = cat_stats["volume_median"] / 1e3
cat_stats["pct_yes"] = (cat_stats["pct_yes"] * 100).round(1)

display_cols = ["n_markets", "volume_total_M", "volume_median_K", "pct_yes"]
print("Breakdown by category:\n")
print(cat_stats[display_cols].to_string(
    float_format=lambda x: f"{x:.1f}",
    header=["Markets", "Total vol ($M)", "Median vol ($K)", "% Yes winner"],
))

In [ ]:
# Sample questions — top 3 by volume in each category
print("Sample markets by category (top 3 by volume):\n")
for cat in sorted(selected["category"].unique()):
    top = selected[selected["category"] == cat].nlargest(3, "volume")
    tickers = CATEGORY_RULES[cat]["tickers"]
    print(f"── {cat} (tickers: {', '.join(tickers)}) ──")
    for _, row in top.iterrows():
        outcome = "Yes" if row["outcome_real"] == 1 else "No"
        print(f"  ${row['volume']/1e6:.1f}M │ {outcome:3s} │ {row['question'][:85]}")
    print()

In [ ]:
# Visualization: number of markets and volume by category
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {
    "fed_rates": "#2196F3",
    "inflation_cpi": "#4CAF50",
    "macro_gdp": "#FF9800",
    "geopolitics_trade": "#F44336",
    "crypto_price": "#9C27B0",
}

cats = cat_stats.index.tolist()
bar_colors = [colors.get(c, "#999") for c in cats]

# Number of markets
axes[0].barh(cats, cat_stats["n_markets"], color=bar_colors, alpha=0.85)
axes[0].set_xlabel("Number of markets")
axes[0].set_title("Selected markets by category")
for i, v in enumerate(cat_stats["n_markets"]):
    axes[0].text(v + 5, i, str(v), va="center", fontsize=10)

# Total volume
axes[1].barh(cats, cat_stats["volume_total_M"], color=bar_colors, alpha=0.85)
axes[1].set_xlabel("Total volume ($M)")
axes[1].set_title("Cumulative volume by category")
for i, v in enumerate(cat_stats["volume_total_M"]):
    axes[1].text(v + 0.5, i, f"${v:.0f}M", va="center", fontsize=10)

fig.tight_layout()
plt.show()

In [ ]:
# Outcome distribution (Yes vs No winner)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Global
outcome_counts = selected["outcome_real"].value_counts().sort_index()
labels = ["No (0)", "Yes (1)"]
axes[0].pie(
    outcome_counts.values, labels=labels, autopct="%1.0f%%",
    colors=["#F44336", "#4CAF50"], startangle=90,
    textprops={"fontsize": 12},
)
axes[0].set_title("Overall outcome distribution")

# By category
cat_outcome = selected.groupby("category")["outcome_real"].value_counts().unstack(fill_value=0)
cat_outcome.plot.bar(
    ax=axes[1], color=["#F44336", "#4CAF50"], alpha=0.85, edgecolor="white",
)
axes[1].set_title("Outcomes by category")
axes[1].set_xlabel("")
axes[1].set_ylabel("Number of markets")
axes[1].legend(["No", "Yes"], loc="upper right")
axes[1].tick_params(axis="x", rotation=25)

fig.tight_layout()
plt.show()

**Observations**:
- **crypto_price** dominates in count (816 markets) — unsurprising since Polymarket is crypto-native and these markets are very granular ("Bitcoin above $X by date Y")
- In volume, **fed_rates** dwarfs everything else (~$4B cumulative) — Fed decisions concentrate the most attention and stakes
- **66% of outcomes are "No"** — prediction markets often ask ambitious questions ("Bitcoin at $1M?", "US recession in 2025?") where the answer is usually no. This is consistent with the **favorite-longshot bias** documented in the literature

## 5. Volume Distribution

Let's see how volume is distributed — this matters because our analysis will be more reliable on the most liquid markets.

In [ ]:
# Volume distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Global histogram
axes[0].hist(
    np.log10(selected["volume"]), bins=30, color="#2196F3", alpha=0.8, edgecolor="white",
)
axes[0].set_xlabel("Volume (log10 USD)")
axes[0].set_ylabel("Number of markets")
axes[0].set_title("Volume distribution (all markets)")
# Reference lines
for val, label in [(5, "$100K"), (6, "$1M"), (7, "$10M"), (8, "$100M")]:
    axes[0].axvline(val, color="gray", linestyle="--", alpha=0.5)
    axes[0].text(val, axes[0].get_ylim()[1] * 0.9, label, fontsize=9, ha="center")

# Box plot by category
data_by_cat = [
    np.log10(selected[selected["category"] == cat]["volume"])
    for cat in sorted(selected["category"].unique())
]
bp = axes[1].boxplot(
    data_by_cat, labels=sorted(selected["category"].unique()),
    patch_artist=True, medianprops=dict(color="black"),
)
sorted_cats = sorted(selected["category"].unique())
for patch, cat in zip(bp["boxes"], sorted_cats):
    patch.set_facecolor(colors.get(cat, "#999"))
    patch.set_alpha(0.7)
axes[1].set_ylabel("Volume (log10 USD)")
axes[1].set_title("Volume by category")
axes[1].tick_params(axis="x", rotation=25)

fig.tight_layout()
plt.show()

## 6. Building Hourly Price Series

Now that we have our markets, we need the **price history** for each one. The CLOB API doesn't provide history for resolved markets — so we use **onchain trades** stored in BigQuery.

For each market and each hour, we compute the **VWAP** (Volume-Weighted Average Price) of the YES token:

$$\text{VWAP}_h = \frac{\sum_{i \in h} \text{price}_i \times \text{volume}_i}{\sum_{i \in h} \text{volume}_i}$$

Hours without trades are filled using **forward-fill** (propagating the last known price), with a 48h limit — beyond that, the market is considered inactive.

In [ ]:
# If we already have the pre-computed file, load it directly.
# Otherwise, run the BigQuery query (next cell).

POLY_HOURLY_PATH = os.path.join(DATA_DIR, "polymarket_hourly.parquet")

if os.path.exists(POLY_HOURLY_PATH):
    poly = pd.read_parquet(POLY_HOURLY_PATH)
    poly["hour_utc"] = pd.to_datetime(poly["hour_utc"], utc=True)
    print(f"Hourly series loaded from cache: {POLY_HOURLY_PATH}")
    print(f"  {poly['condition_id'].nunique()} markets, {len(poly):,} observations")
    print(f"  Period: {poly['hour_utc'].min().date()} -> {poly['hour_utc'].max().date()}")
    print(f"  Interpolated hours (forward-fill): {poly['is_interpolated'].mean()*100:.1f}%")
else:
    print("File polymarket_hourly.parquet not found.")
    print("Run the BigQuery cell below to generate it.")

In [ ]:
# ── BigQuery query (run only if polymarket_hourly.parquet doesn't exist) ──
# Uncomment and run if needed.

# from google.cloud import bigquery
#
# BQ_PROJECT = "polymarket-research-490517"
# MAX_FFILL_HOURS = 48
#
# client = bigquery.Client(project=BQ_PROJECT)
# condition_ids = selected["condition_id"].unique().tolist()
#
# query = """
# SELECT
#     t.condition_id,
#     TIMESTAMP_TRUNC(TIMESTAMP_SECONDS(CAST(t.timestamp AS INT64)), HOUR) AS hour_utc,
#     SUM(t.price * t.usd_amount) / NULLIF(SUM(t.usd_amount), 0) AS vwap_yes,
#     SUM(t.usd_amount) AS volume_usd,
#     COUNT(*) AS trade_count
# FROM polymarket.trades t
# WHERE t.condition_id IN UNNEST(@cids)
#   AND t.nonusdc_side = 'token1'
# GROUP BY t.condition_id, hour_utc
# ORDER BY t.condition_id, hour_utc
# """
#
# job_config = bigquery.QueryJobConfig(
#     query_parameters=[
#         bigquery.ArrayQueryParameter("cids", "STRING", condition_ids),
#     ]
# )
#
# print(f"Querying BigQuery for {len(condition_ids)} markets...")
# raw = client.query(query, job_config=job_config).to_dataframe()
# print(f"  -> {len(raw):,} rows, {raw['condition_id'].nunique()} markets with trades")
#
# # Forward-fill gaps (max 48h)
# results = []
# for cid, group in raw.groupby("condition_id"):
#     group = group.sort_values("hour_utc")
#     full_idx = pd.date_range(
#         start=group["hour_utc"].min(), end=group["hour_utc"].max(),
#         freq="h", tz="UTC",
#     )
#     group = group.set_index("hour_utc").reindex(full_idx)
#     group.index.name = "hour_utc"
#     group["condition_id"] = cid
#     group["is_interpolated"] = group["vwap_yes"].isna()
#     group["vwap_yes"] = group["vwap_yes"].ffill(limit=MAX_FFILL_HOURS)
#     group["volume_usd"] = group["volume_usd"].fillna(0)
#     group["trade_count"] = group["trade_count"].fillna(0).astype(int)
#     group = group.dropna(subset=["vwap_yes"])
#     results.append(group.reset_index())
#
# poly = pd.concat(results, ignore_index=True)
# poly.to_parquet(POLY_HOURLY_PATH, index=False)
# print(f"Saved -> {POLY_HOURLY_PATH}")
# print(f"  {poly['condition_id'].nunique()} markets, {len(poly):,} obs")

## 7. Exploring Price Series

Let's visualize some price series to verify data consistency and understand what a Polymarket market looks like over time.

In [ ]:
# Per-market stats
market_stats = (
    poly.groupby("condition_id")
    .agg(
        n_hours=("hour_utc", "count"),
        first_date=("hour_utc", "min"),
        last_date=("hour_utc", "max"),
        pct_interpolated=("is_interpolated", "mean"),
        total_volume=("volume_usd", "sum"),
        avg_price=("vwap_yes", "mean"),
    )
)

print(f"Statistics across {len(market_stats)} price series:\n")
print(f"  Median duration      : {market_stats['n_hours'].median() / 24:.0f} days")
print(f"  Max duration         : {market_stats['n_hours'].max() / 24:.0f} days")
print(f"  % interpolated (median): {market_stats['pct_interpolated'].median()*100:.0f}%")
print(f"  Median volume        : ${market_stats['total_volume'].median():,.0f}")

In [ ]:
# Plot sample price series — one market per category (highest volume)
enriched = poly.merge(
    selected[["condition_id", "category", "question", "outcome_real"]],
    on="condition_id", how="left",
)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, cat in enumerate(sorted(selected["category"].unique())):
    ax = axes[idx]
    # Top market by volume in this category
    top_cid = (
        selected[selected["category"] == cat]
        .nlargest(1, "volume")["condition_id"]
        .iloc[0]
    )
    series = enriched[enriched["condition_id"] == top_cid].sort_values("hour_utc")
    question = series["question"].iloc[0] if not series.empty else cat
    outcome = series["outcome_real"].iloc[0] if not series.empty else "?"

    ax.plot(series["hour_utc"], series["vwap_yes"], color=colors.get(cat, "#999"),
            linewidth=1.2, alpha=0.9)

    # Mark interpolated hours
    interp = series[series["is_interpolated"]]
    if len(interp) > 0 and len(interp) < len(series) * 0.8:
        ax.scatter(interp["hour_utc"], interp["vwap_yes"],
                   color="gray", s=1, alpha=0.3, label="interpolated")

    ax.set_ylim(-0.02, 1.02)
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.4)
    # Resolution line
    outcome_label = "Yes" if outcome == 1 else "No"
    ax.axhline(outcome, color="green" if outcome == 1 else "red",
               linestyle="--", alpha=0.4, label=f"Outcome: {outcome_label}")

    ax.set_title(f"{cat}\n{question[:70]}...", fontsize=10)
    ax.set_ylabel("Yes price ($)")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

# Hide 6th subplot if only 5 categories
if len(selected["category"].unique()) < 6:
    axes[5].set_visible(False)

fig.suptitle("Sample price series — one market per category", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 8. Corresponding Financial Data

For each Polymarket market category, we collected prices of associated financial assets via **yfinance** — 14 tickers total, hourly and daily data over 2 years.

In [ ]:
# Financial data
fin_h = pd.read_csv(os.path.join(DATA_DIR, "financial_hourly.csv"))
fin_d = pd.read_csv(os.path.join(DATA_DIR, "financial_daily.csv"))
fin_h["datetime"] = pd.to_datetime(fin_h["datetime"], utc=True)
fin_d["datetime"] = pd.to_datetime(fin_d["datetime"], utc=True)

print(f"Hourly data: {len(fin_h):,} obs, {fin_h['ticker'].nunique()} tickers")
print(f"Daily data:  {len(fin_d):,} obs")
print(f"Period: {fin_h['datetime'].min().date()} -> {fin_h['datetime'].max().date()}")
print(f"\nTickers: {sorted(fin_h['ticker'].unique())}")

# Observations per ticker
ticker_counts = fin_h.groupby("ticker").size().sort_values(ascending=False)
print(f"\nObservations per ticker (hourly):")
for ticker, count in ticker_counts.items():
    desc = fin_h[fin_h["ticker"] == ticker]["description"].iloc[0] if "description" in fin_h.columns else ""
    print(f"  {ticker:12s} | {count:>6,} obs | {desc}")

In [ ]:
# Financial asset overview — normalized to base 100 for comparison
fig, ax = plt.subplots(figsize=(14, 6))

key_tickers = ["SPY", "TLT", "GLD", "BTC-USD", "^VIX"]
ticker_colors = {
    "SPY": "#2196F3", "TLT": "#4CAF50", "GLD": "#FF9800",
    "BTC-USD": "#9C27B0", "^VIX": "#F44336",
}

for ticker in key_tickers:
    t_data = fin_d[fin_d["ticker"] == ticker].sort_values("datetime")
    if t_data.empty:
        continue
    # Normalize to base 100
    base = t_data["close"].iloc[0]
    ax.plot(
        t_data["datetime"], t_data["close"] / base * 100,
        label=ticker, color=ticker_colors.get(ticker, "#999"),
        linewidth=1.2, alpha=0.85,
    )

ax.set_ylabel("Normalized price (base 100)")
ax.set_title("Key financial assets — 2 years (base 100)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Save and Summary

In [ ]:
# Save selected markets
output_path = os.path.join(DATA_DIR, "selected_markets.csv")
selected.to_csv(output_path, index=False)
print(f"Markets saved -> {output_path}")

# Final summary
print(f"\n{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Selected markets       : {len(selected)}")
print(f"  Categories             : {selected['category'].nunique()}")
print(f"  Hourly series          : {len(poly):,} observations")
print(f"  Financial tickers      : {fin_h['ticker'].nunique()}")
print(f"  Polymarket period      : {poly['hour_utc'].min().date()} -> {poly['hour_utc'].max().date()}")
print(f"  Financial period       : {fin_h['datetime'].min().date()} -> {fin_h['datetime'].max().date()}")
print(f"\nOutput files:")
print(f"  data/selected_markets.csv       ({len(selected)} markets)")
print(f"  data/polymarket_hourly.parquet   ({len(poly):,} obs)")
print(f"  data/financial_hourly.csv        ({len(fin_h):,} obs)")
print(f"  data/financial_daily.csv         ({len(fin_d):,} obs)")

## What We Have Now

We've built a clean dataset of **1,083 macro-financial markets** across 5 categories, with their hourly price series (~879K observations) and corresponding financial asset data (14 tickers, 2 years).

**Next step** -> [Notebook 02: Crowd Calibration](02_crowd_calibration.ipynb) — prove that Polymarket is a reliable signal by measuring its probabilistic accuracy via the Brier Score.